<a href="https://colab.research.google.com/github/JaneCS2025/GenAI/blob/main/Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:3b"     # small and fast. For agent lessons, "qwen2.5:7b" is smarter (slower to download)
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

✅ Ollama is running with qwen2.5:3b


In [9]:
!pip -q install langchain langchain-ollama langchain-core
import warnings; warnings.filterwarnings('ignore')
from langchain_ollama import ChatOllama
llm = ChatOllama(model=MODEL, temperature = 0.3)

reply = llm.invoke("What is the weather in Vancouver, BC today")
print(type(reply).__name__, '->', reply.content)

AIMessage -> I don't have real-time access to current weather data, including the weather in Vancouver, BC. To get the most accurate and up-to-date information on the weather in Vancouver, I recommend checking a reliable weather website or app, such as the Weather Channel, AccuWeather, or the official website of the National Weather Service. They provide current weather conditions, forecasts, and alerts for various locations, including Vancouver, BC.


In [4]:
from langchain_core.prompts import PromptTemplate
template = "Give me 3 careers skills that are in high demand in {year}"
promt_template = PromptTemplate.from_template(template)

In [5]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [6]:
chain = promt_template | llm | parser

In [10]:
response = chain.invoke({'year': '2026'})
print("\n Career skills in 2026:\n", response)


 Career skills in 2026:
 Certainly! Here are three career skills that are expected to be in high demand in 2026, based on current trends and predictions:

1. **Data Science and Analytics**: With the increasing amount of data being generated across various industries, there is a growing need for professionals who can effectively analyze and interpret this data. Skills in data mining, machine learning, predictive analytics, and data visualization will be particularly in demand.

2. **Artificial Intelligence and Machine Learning**: As AI and machine learning technologies continue to advance, there will be a need for professionals who can design, develop, and implement AI solutions. This includes skills in natural language processing, computer vision, and deep learning. Additionally, understanding how to integrate AI into existing systems and ensuring ethical considerations will also be crucial.

3. **Cybersecurity**: With the increasing reliance on digital systems and the rise of cyber t

In [11]:
response = chain.invoke({'year': '2025'})
print("\n Career skills in 2025:\n", response)


 Career skills in 2025:
 In 2025, several skills are expected to be in high demand across various industries. Here are three such skills:

1. **Data Science and Analytics**: With the increasing amount of data being generated and the need for data-driven decision-making, professionals skilled in data science and analytics will be in high demand. This includes skills in data collection, data cleaning, data analysis, predictive modeling, and the interpretation of complex data sets to derive actionable insights.

2. **Artificial Intelligence and Machine Learning**: As AI and machine learning technologies continue to evolve, there will be a growing need for professionals who can design, develop, and implement AI and ML solutions. This includes skills in natural language processing, computer vision, deep learning, and the ability to integrate AI into existing systems to automate processes and improve efficiency.

3. **Cybersecurity**: With the increasing number of cyber threats and the grow

In [12]:
# Prompt Templates 1
from langchain_core.messages import SystemMessage, HumanMessage
print(llm.invoke([SystemMessage("You only answer in rhyming couplets."),
                  HumanMessage("What is a volcano?")]).content)


A volcano is a fiery beast,  
That spews hot ash and molten best.


In [14]:
#Prompt Templates 2
from langchain_core.prompts import ChatPromptTemplate
explain = ChatPromptTemplate.from_messages([
    ("system", "You are a {subject} teacher. Explain things for a {age}-year-old in at most {sentences} sentences."),
    ("user", "{question}")
])
filled = explain.invoke({"subject": "biology", "age": 12, "sentences": 2, "question": "what is a cell"})
print(filled.to_messages())

[SystemMessage(content='You are a biology teacher. Explain things for a 12-year-old in at most 2 sentences.', additional_kwargs={}, response_metadata={}), HumanMessage(content='what is a cell', additional_kwargs={}, response_metadata={})]


In [15]:
parser = StrOutputParser()
chain = explain | llm | parser
print(chain.invoke({"subject": "biology", "age": 12, "sentences": 2, "question": "what is a cell"}))

A cell is like the smallest building block of life. It's the basic unit that makes up living things and performs important functions like eating, growing, and repairing itself.


In [17]:
questions = [{"subject": "biology", "age": 12, "sentences": 2, "question": q}
             for q in ["what is normal heart rate?", "What is normal blood pressure?", "what is normal cholesterol level?"]]
for q, a in zip(questions, chain.batch(questions)):
  print(f" ? {q['question']}\n {a}")


 ? what is normal heart rate?
 A normal resting heart rate for a 12-year-old is usually between 60 to 100 beats per minute when they are not active.
 ? What is normal blood pressure?
 Normal blood pressure for an adult is typically considered to be less than 120 over 80 (written as 120/80 mmHg). For children, the numbers are lower and can vary depending on their age and height.
 ? what is normal cholesterol level?
 Normal cholesterol levels are typically below 200 milligrams per deciliter (mg/dL) of blood, but a healthy level is usually below 180 mg/dL. This is important for keeping your heart healthy.


In [21]:
print("n\Streaming: ", end ="")
for token in chain.stream({"subject": "geography", "age": 12, "sentences": 5, "question": "What is glacier?"}):
    print(token, end="", flush=True)


n\Streaming: A glacier is a big, thick ice sheet that forms in very cold places like Antarctica or the Arctic. It moves slowly over time, slowly shaping the land as it goes. Imagine a really, really big pile of snow that never melts, and that's what a glacier is!

In [23]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
helper = RunnableParallel(
    simple=ChatPromptTemplate.from_template("Explain {idea} to a 7-year-old in one sentence.") | llm | StrOutputParser(),
    expert=ChatPromptTemplate.from_template("Explain {idea} to a university student in one sentence.") | llm | StrOutputParser(),
    emoji=ChatPromptTemplate.from_template("Summarise {idea} using only 5 emojis.") | llm | StrOutputParser(),
)

for k,v in helper.invoke({"idea": "photosynthesis"}).items():
  print(f"{k:7}:{v}")

simple :Photosynthesis is like a magic recipe that plants use to make their food using sunlight, water, and air, turning them into energy to grow bigger!
expert :Photosynthesis is the process by which plants convert light energy into chemical energy, storing it in the bonds of glucose molecules, using carbon dioxide and water to produce oxygen as a byproduct.
emoji  : Leafs 🌱 🌻
